# Proyecto EDA: Análisis exploratorio de datos COVID-19 en Estados Unidos

## 1. Introducción

El objetivo de este proyecto es realizar un análisis exploratorio de datos (EDA) sobre la evolución de la pandemia de COVID-19 en Estados Unidos.

A través de Python y sus principales librerías de análisis de datos, se estudiará la evolución de los casos confirmados, las pruebas realizadas y los fallecimientos registrados en los diferentes estados.

El análisis permitirá identificar tendencias temporales, diferencias entre estados y posibles patrones relacionados con el impacto de la pandemia.

## 2. Objetivos

- Analizar la estructura y calidad del conjunto de datos.
- Identificar y tratar valores faltantes, duplicados e inconsistencias.
- Estudiar la evolución temporal de los principales indicadores.
- Comparar el impacto de la pandemia entre estados.
- Crear visualizaciones que faciliten la interpretación de los resultados.
- Comunicar los principales hallazgos.

### 1.1. Preguntas de investigación

Antes de analizar los datos nos planteamos estas tres preguntas:

1. **Evolución temporal:** ¿cómo evolucionaron los casos nuevos y los fallecimientos durante el periodo?
2. **Comparación entre estados:** ¿qué estados concentraron más casos positivos?
3. **Relación pruebas–casos:** ¿existe relación entre el número de pruebas realizadas y los casos positivos?

Estas preguntas se responden en la sección 7.

## 2. Cargar los datos

En esta sección descargamos el dataset desde la API pública del *COVID Tracking Project* y lo dejamos preparado para el análisis.

### 2.1. Importar librerías

Primero, importamos las librerías necesarias para la descarga, manipulación y visualización de los datos.

In [ ]:
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

### 2.2. Descargar o leer el dataset

Para no descargar los datos en cada ejecución, el notebook lee el CSV local si ya existe. Si no existe, descarga los datos desde la API y los guarda en `data/covid_us.csv`.

In [ ]:
URL = "https://api.covidtracking.com/v1/states/daily.json"
DATA_PATH = Path("../data/covid_us.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Datos cargados desde {DATA_PATH}")
else:
    response = requests.get(URL, timeout=30)
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print("Datos descargados desde la API")

df.head()

## 3. Revisión de la estructura del dataset

Antes de limpiar los datos, revisamos su tamaño, los tipos de cada columna y las primeras filas para entender qué información contienen.

In [ ]:
# Número de filas y columnas
df.shape

In [ ]:
# Información general
df.info()

In [ ]:
# Primeras filas
df.head()

In [ ]:
# Estadísticas descriptivas
df.describe()

El dataset contiene **20.780 registros y 56 columnas**: un registro por estado o territorio y día, entre el **13 de enero de 2020** y el **7 de marzo de 2021** (56 estados/territorios en total).

`info()` muestra que la mayoría de las columnas son numéricas, pero `date` viene como entero (`20210307`) y hay muchas columnas con gran cantidad de valores nulos. `describe()` confirma que las medidas acumuladas están muy sesgadas: la media de casos positivos es mucho mayor que la mediana, porque unos pocos estados grandes concentran gran parte de los casos.

## 4. Limpieza de datos

En esta sección identificamos duplicados y valores faltantes, corregimos los tipos de datos y seleccionamos las columnas que utilizaremos en el análisis. Cada decisión se justifica en el texto.

### 4.1. Duplicados

Comprobamos si hay filas repetidas y si se repite la combinación de estado y fecha, que debería ser única.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("Duplicados por estado y fecha:", df.duplicated(subset=["state", "date"]).sum())

No hay filas duplicadas ni duplicados de `(estado, fecha)`. Por tanto, cada observación es única y no es necesario eliminar registros.

### 4.2. Valores faltantes

Calculamos el número y el porcentaje de nulos por columna.

In [ ]:
nulos = df.isna().sum()
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(df) * 100).round(1),
}).sort_values("porcentaje", ascending=False)

resumen_nulos.head(15)

La tabla muestra que **14 columnas superan el 80 % de nulos** (por ejemplo `pending`, `inIcuCumulative`, `onVentilatorCumulative`, `hospitalizedDischarged` y las de anticuerpos y antígenos). Además, `dataQualityGrade` y `grade` están **vacías al 100 %**.

Imputar columnas con tan poca información añadiría ruido sin aportar valor. Por eso decidimos **seleccionar un subconjunto de variables relevantes** y descartar el resto.

### 4.3. Selección y renombrado de columnas

Nos quedamos con 10 columnas relacionadas con las preguntas de investigación: fecha, estado, casos, pruebas, fallecimientos y hospitalizaciones. Las renombramos al español para que el notebook sea más legible.

In [ ]:
columnas = {
    "date": "fecha",
    "state": "estado",
    "positive": "casos_positivos",
    "negative": "pruebas_negativas",
    "death": "fallecimientos",
    "totalTestResults": "pruebas_totales",
    "hospitalizedCurrently": "hospitalizados_actuales",
    "positiveIncrease": "casos_nuevos",
    "deathIncrease": "fallecimientos_nuevos",
    "totalTestResultsIncrease": "pruebas_nuevas",
}

df = df[list(columnas)].rename(columns=columnas)
df.head()

Eliminamos 46 columnas: metadatos que no aportan al análisis (`hash`, `fips`), puntuaciones internas del proyecto (`score`, `grade`, `commercialScore`, etc.) y variables con más del 74 % de nulos. Conservar solo las variables relevantes hace que el análisis sea más claro y reproducible.

### 4.4. Corrección de tipos de datos

`fecha` está almacenada como un entero con formato `AAAAMMDD`; la convertimos a fecha real. También convertimos `estado` a variable categórica, porque tiene un número limitado de valores repetidos.

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"].astype(str), format="%Y%m%d")
df["estado"] = df["estado"].astype("category")

# Ordenamos por estado y fecha para poder rellenar series temporales
df = df.sort_values(["estado", "fecha"]).reset_index(drop=True)

df.dtypes

### 4.5. Tratamiento de los valores faltantes

Las columnas acumuladas (`casos_positivos`, `pruebas_negativas`, `fallecimientos` y `pruebas_totales`) son series temporales por estado. Rellenamos sus nulos con el último valor conocido del mismo estado (`ffill`), ya que un acumulado no puede disminuir y el dato anterior es la mejor aproximación disponible.

`hospitalizados_actuales` es una medida instantánea, no un acumulado; por eso **no la imputamos** y mantenemos sus nulos.

In [ ]:
columnas_acumuladas = [
    "casos_positivos",
    "pruebas_negativas",
    "fallecimientos",
    "pruebas_totales",
]

df[columnas_acumuladas] = (
    df.groupby("estado", observed=True)[columnas_acumuladas].ffill()
)

In [ ]:
df[
    [
        "casos_positivos",
        "pruebas_negativas",
        "fallecimientos",
        "pruebas_totales",
        "hospitalizados_actuales",
    ]
].isna().sum()

Tras el relleno, los nulos restantes en las columnas acumuladas están al principio de la serie de cada estado (antes de su primer dato reportado), por lo que no hay un valor anterior que utilizar. No los eliminamos para no perder registros; las visualizaciones descartan esos valores de forma automática.

### 4.6. Inconsistencias: incrementos negativos

Algunos estados notificaron incrementos diarios negativos. No son errores del análisis: son **correcciones retroactivas** que hicieron los estados sobre sus acumulados. Las documentamos y las mantenemos, ya que eliminarlas alteraría la serie original.

In [ ]:
for col in ["casos_nuevos", "fallecimientos_nuevos", "pruebas_nuevas"]:
    print(f"{col}: {(df[col] < 0).sum()} valores negativos")

### 4.7. Guardar el dataset limpio

Guardamos una copia del dataset limpio para poder reutilizarlo. Este archivo también está excluido del control de versiones.

In [ ]:
df.to_csv("../data/covid_us_clean.csv", index=False)
print(f"Dataset limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 5. Análisis univariado

Analizamos tres variables por separado: los casos positivos acumulados, la cobertura de registros por estado y los fallecimientos acumulados.

### 5.1. Distribución de los casos positivos acumulados

Como los acumulados abarcan varios órdenes de magnitud, usamos una escala logarítmica en el eje X para que la forma de la distribución sea visible.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    df["casos_positivos"].dropna(),
    bins=40,
    log_scale=True,
    color="steelblue",
    ax=ax,
)
ax.set_title("Distribución de casos positivos acumulados")
ax.set_xlabel("Casos positivos acumulados (escala logarítmica)")
ax.set_ylabel("Número de registros")
plt.tight_layout()
plt.show()

La distribución es **asimétrica a la derecha**: la mayoría de los registros se concentran en valores bajos y una cola larga recoge los estados con más casos. Esto confirma que comparar totales sin normalizar favorece a los estados más grandes.

### 5.2. Número de registros por estado

Comprobamos cuántos días aporta cada estado. Una cobertura desigual podría sesgar las comparaciones temporales.

In [ ]:
registros = df["estado"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 12))
sns.barplot(
    x=registros.values,
    y=registros.index.astype(str),
    color="seagreen",
    ax=ax,
)
ax.set_title("Número de registros por estado")
ax.set_xlabel("Número de registros")
ax.set_ylabel("Estado")
plt.tight_layout()
plt.show()

Cada estado aporta entre **357 y 420 registros**, una cobertura parecida pero no idéntica. Las diferencias se deben a que algunos territorios empezaron a reportar más tarde o dejaron de hacerlo antes del final del periodo.

### 5.3. Distribución de los fallecimientos acumulados

El boxplot resume la mediana, la dispersión y los valores atípicos de los fallecimientos acumulados.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    x=df["fallecimientos"].dropna(),
    color="salmon",
    ax=ax,
)
ax.set_title("Distribución de fallecimientos acumulados")
ax.set_xlabel("Fallecimientos acumulados")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

La caja se concentra en valores bajos y aparecen numerosos **valores atípicos** a la derecha. Son estados grandes con muchos fallecimientos acumulados; no los eliminamos porque son observaciones reales y relevantes, no errores de medición.